# BP8 Gate 1 — Business Understanding & Policy
**Customer360 Navigator Enterprise Suite — Executive/Product Analytics**

## Why this notebook exists, and what BP8 actually is
Master Execution Plan Section 5's BP table marks BP8 `Integrates BANKING77? = YES (consumes
outputs)` — deliberately different from BP1/BP2/BP4's direct `YES` and BP6's own `YES`. BP8 never
touches BANKING77 (or CFPB) directly; Section 5.1 states its job in one sentence: **"aggregate
Gold-layer outputs from BP1–BP7 into executive/product KPIs, drill-downs, friction trends,
escalation trends and product opportunity flags in the Power BI layer (Section 19)."** Section 6's
architecture diagram draws this literally: every upstream BP's outputs converge into one final
arrow, `All analytical outputs -> BP8 Power BI`. BP8 is the suite's cross-BP rollup layer, not an
eighth modeling problem — Section 17.5 confirms this explicitly by giving BP8 its own tech-stack
row, `Power BI Gold layer / Gold-table build script`, with **no classifier benchmark and no GenAI
call**, closest in spirit to BP4's own aggregation-pipeline approach (Polars-based), though BP8's
real job is cross-BP rollup, not single-BP journey analytics. This notebook is BP8's very first —
Gate 1 defines BP8's scope and policy before any Gate 2 aggregation pipeline is written, exactly as
Section 8 requires for every BP ("every one of the 8 Business Problems moves through the same six
gates, in order").

## The "Power BI-ready" scope, clarified from Section 19 — not assumed
Section 19 ("Power BI / Decision Intelligence") says, in full: *"Build Gold/semantic tables in
Python, Parquet/CSV or SQL-ready form; create the interactive .pbix in Power BI Desktop."* Read
together with Section 18.2's repository layout (`powerbi/{gold_tables, pbix}`) and Section 17.5's
optimization levers ("vectorized Parquet/CSV writes; pre-aggregate in Python rather than relying on
Power BI-side heavy transforms"), this is unambiguous: **BP8's own Claude-authored deliverable is
the Gold/semantic table build (Python, writing to `powerbi/gold_tables/`) — the `.pbix` file itself
is built by you, in the Power BI Desktop application, not authored directly by Claude.** This
notebook does not assume that split — it is live-verified below (Section 5): both `powerbi/gold_tables/`
and `powerbi/pbix/` already exist as folders in this project's locked structure (Section 18.2 /
`PROJECT_STRUCTURE_LOCKED.md`) and, checked live, both are currently empty — nothing has been built
in either yet. BP8's own scope, defined at this Gate 1, is therefore bounded to the Python-built
Gold-table layer; the `.pbix` artifact is explicitly named as a downstream, human, Power-BI-Desktop
step, out of this notebook's and this suite's Claude-authored scope, never claimed as delivered here.

## The distinction this notebook exists partly to prevent confusing: BP1–4's own Gate 7 vs. BP8
This suite already has a real, per-BP "Executive Rollup Report" pattern: BP1, BP2, BP3 and BP4 each
built their **own** `bp{n}_..._g7_executive_rollup_report.ipynb` producing one HTML dashboard, one
DOCX report, one XLSX workbook and one PPTX deck — real files, live-verified to exist on disk below
(Section 6) — that package **that single BP's own** Gates 1–6 evidence for a recruiter/portfolio
audience. Master Plan Section 20 names this the suite's **optional** Financial-Impact/recruiter
reporting layer ("Trigger this layer explicitly per-BP; it is not assumed for all eight"), separate
from Section 5/6/8/17.5/19's **required** BP8 Power BI decision layer. **These are two different
things, stated here explicitly so no future gate or reader conflates them:**
- **BP1–BP4's own Gate 7 rollups** (already real, already built, per-BP, single-audience: HTML +
  DOCX + XLSX + PPTX under each BP's own `reports/bp{n}_.../executive_rollup/`) — **not BP8's job,
  not touched by this notebook or any future BP8 gate.**
- **BP8's own Power BI Gold layer** (new, cross-BP, Power-BI-audience: Gold/semantic Parquet/CSV
  tables under `powerbi/gold_tables/`, feeding a `.pbix` you build in Power BI Desktop) — **BP8's
  actual, sole job**, not yet started (live-verified below), this notebook's own Gate 1 scope.

## The "no predictive target" nuance, addressed honestly rather than forced
Section 8's Gate 1 row defines the generic gate output as `policy.json: target definition, leakage
rules, ASSUMPTIONs` with exit criterion `"No target leakage possible by construction"`. BP8 trains
no model and predicts nothing — it aggregates already-computed Gold-layer columns from upstream BPs
into KPI tables. Forcing BP1–BP3's `target_definition` schema onto a BP with no target would be
dishonest scaffolding, not policy. This notebook states plainly, as a real policy field
(`no_predictive_target: true`), that BP8 has no predictive target, so Section 8's own leakage exit
criterion is **trivially satisfied by construction** — there is no target for any feature to leak
into. `policy.json` instead captures what BP8 actually needs at Gate 1: which upstream Gold-layer
sources are in scope today vs. pending, what KPI categories those real sources support, what is
explicitly out of scope, and the data-minimization statement — the same pattern BP4's own Gate 1
already established for its own targetless scope (`journey_definition` in place of
`target_definition`; this notebook's analog is `aggregation_scope_definition`).

## What this notebook live-verifies, and why
Rather than assuming which of BP1–BP7's outputs are real today, this notebook checks, live, against
this project's own files:
1. Each of BP1–BP7's own `configs/bp{n}_....yaml` `status` field (real, on-disk, not asserted from
   memory or from this session's own prior conversation).
2. Each of BP1–BP7's own `notebooks/bp{n}_.../artifacts/policy.json` existence.
3. The real Gold-layer Parquet tables in `data/processed/` that a completed upstream BP has actually
   written — read live via `pl.scan_parquet` (row count + real column list), never assumed from a
   config file's own claim.
4. Each of BP1–BP4's own Gate 7 executive-rollup output files (existence + size only — this notebook
   never opens or parses another BP's DOCX/XLSX/PPTX; that is that BP's own Gate 7 job, not BP8's).
5. `powerbi/gold_tables/` and `powerbi/pbix/` — confirming, live, that both are currently empty.

This inventory is what makes BP8's Gate 1 policy **forward-compatible by construction**: BP8's real
KPI scope (Section 9 below) is grounded only in the upstream Gold-layer columns that live-verified
to actually exist today (BP1, BP2, BP3, BP4); BP5 (root-cause/driver analytics), BP6 (GenAI
resolution) and BP7 (decision engine) are recorded honestly as `not_started` (live-verified, no
Gold-layer output exists for any of them yet) — their own KPI categories are explicitly deferred,
never guessed at or stubbed with placeholder field names now.

## Real KPI scope, grounded in real upstream fields — not invented in the abstract
Section 5.1's own words for BP8 name four KPI categories: "executive/product KPIs, drill-downs,
friction trends, escalation trends and product opportunity flags." This notebook grounds each one in
a real, already-delivered upstream field, live-verified in Section 6/7 below:
- **Friction trends** — BP2's own real `friction_severity_class` column
  (`data/processed/cfpb_friction_severity_gold.parquet`), the 4-class ordinal severity taxonomy
  (`LOW_FRICTION` / `MEDIUM_FRICTION` / `MEDIUM_HIGH_FRICTION` / `HIGH_FRICTION`, plus
  `EXCLUDED_PENDING` / `EXCLUDED_UNKNOWN`) BP2's own Gate 2 built
  (`configs/bp2_friction_severity_taxonomy.yaml`), trended over `Date received`.
- **Escalation trends** — BP3's own real `intervention_required` binary column
  (`data/processed/cfpb_intervention_escalation_gold.parquet`), trended over `Date received`,
  reported with BP3's own real class balance (10,511 positive / 804,942 negative / 233,122 excluded
  — never a bare accuracy figure, per Master Plan Section 14).
- **Product opportunity flags** — BP4's own real issue-cluster tier rollup
  (`data/processed/cfpb_issue_cluster_summary_gold.parquet` plus BP4's own Gate 5
  `HIGH`/`MEDIUM`/`LOW`/`NONE` review-priority tiers, already real-recorded in
  `configs/bp4_customer_journey_analytics.yaml`'s own `tier_rollup` block) — BP8 aggregates BP4's
  already-computed tiers, it never recomputes BP4's own flags.
- **Customer intent / volume KPIs** — BP1's own real `category` (77-class) and
  `common_taxonomy_bucket` (9-bucket) columns (`data/processed/cfpb_common_taxonomy_gold.parquet`,
  `data/processed/banking77_common_taxonomy_gold.parquet`), including BP1's own real champion-model
  metric (`logistic_regression`, held-out test F1-macro 0.8221, from BP1's own
  `configs/bp1_customer_intent_classification.yaml`).
- **Explicitly deferred, not guessed at**: any KPI category sourced from BP5 (root-cause drivers),
  BP6 (GenAI resolution evidence) or BP7 (priority/decision-engine scores) — live-verified below to
  not exist yet. BP8's own future Gate 2 must re-check these same live statuses before assuming any
  of them are ready, never trust this Gate 1's snapshot as still current by then.

## Purpose
Produces BP8's Gate 1 output exactly as Section 8 defines the generic gate 1 contract, adapted
honestly for a BP with no predictive target: a policy artifact (`policy.json`) recording BP8's
aggregation scope, the real Power BI artifact-format split (Section 19), the BP1–4-rollup-vs-BP8
distinction, the live upstream-readiness inventory, and ASSUMPTIONs — verified against this
project's real, on-disk files, not invented.

## Standing rules this notebook follows
- **Execution boundary** (Section 12.2): Claude wrote this notebook; it does not run it. You run it
  on your own machine, and the real, live-checked upstream-inventory and Gold-layer results below
  become this project's Gate 1 policy record for BP8.
- **Zero-fabrication** (Section 12.1): every status, row count and column list below is read live
  from this project's own real files — no upstream BP's completion status or Gold-table schema is
  asserted from this notebook's own markdown prose alone.
- **Data-minimization & purpose-limitation (GLBA/GDPR-aligned, Section 8/9's Gate 1 compliance
  touchpoint)**: BP8 reads only already-aggregated, already-governed Gold-layer columns BP1–BP7's
  own gates already cleared (severity classes, intervention flags, tier labels, intent buckets) —
  never CFPB/BANKING77 raw rows, never a narrative-text field, never a demographic-adjacent field
  (`Tags`, `ZIP code`) directly. No new data is collected; BP8 re-aggregates what upstream BPs
  already computed and already documented.
- **WARP**: `configure_performance()` first. Every Gold-layer table this notebook reads is scanned
  lazily via `pl.scan_parquet` (schema + row count only, no full materialization) — Gate 1 profiles
  what exists, it does not build the Gold KPI tables themselves (that is BP8 Gate 2's job).
- **HYPER**: reuses `src/utils/bp1_config_sync.py` unmodified (already fully generic, already reused
  unmodified by BP2/BP3/BP4) for BP8's own config file — no BP8-specific config-sync logic written.
- **Idempotent**: re-running this notebook overwrites `configs/bp8_executive_product_analytics.yaml`
  (front matter only) and this notebook's own `policy.json` artifact in place.
- **PROJECT_STRUCTURE_LOCKED.md rule #3**: same project-root resolver as every other notebook in
  this project.

## Outputs (both written, idempotent overwrite-in-place)
- `configs/bp8_executive_product_analytics.yaml` — `aggregation_scope_definition`,
  `scope_boundaries`, `assumptions`, `status` written to the front-matter section (no gate blocks
  exist yet for BP8, so none are at risk of being overwritten)
- `notebooks/bp8_executive_product_analytics/artifacts/policy.json` — the Section 8 Gate 1 output
  artifact, adapted for a targetless aggregation BP, with the full live upstream inventory embedded

## Prerequisites
None of BP8's own upstream gates need to have run for this Gate 1 to execute — Gate 1's job is to
define scope and policy, live-checking what already exists, not to require it. `01_data_acquisition_
profiling.ipynb` should have been real-run at least once so `data/` exists at all.

## If a structural check below fails
It raises `AssertionError` with the failing check named. A failing "no fabricated KPI category"
check in particular must never be worked around by inventing a plausible-sounding BP5/6/7 KPI field
now — the honest response is to leave that KPI category deferred until the real upstream Gold table
exists, exactly as this notebook does for BP5/6/7 today.

In [ ]:
# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
import os
import sys
from pathlib import Path


def _find_project_root(marker_filename: str = "PROJECT_STRUCTURE_LOCKED.md") -> Path:
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        candidate = Path(env_override)
        if (candidate / marker_filename).exists():
            return candidate
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {candidate} but {marker_filename} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    current = start
    for _ in range(8):
        if (current / marker_filename).exists():
            return current
        if current.parent == current:
            break
        current = current.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker_filename in filenames:
            return Path(depth_root)

    raise RuntimeError(
        "Could not resolve PROJECT_ROOT. Set the C360_PROJECT_ROOT environment variable to the "
        "Customer360_Navigator_Enterprise_Suite folder, or run this notebook from inside the project tree "
        "(expected at notebooks/bp8_executive_product_analytics/)."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"
POWERBI_GOLD_TABLES_DIR = PROJECT_ROOT / "powerbi" / "gold_tables"
POWERBI_PBIX_DIR = PROJECT_ROOT / "powerbi" / "pbix"
ARTIFACTS_DIR = NOTEBOOKS_DIR / "bp8_executive_product_analytics" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP - configure_performance() FIRST, before any heavy/BLAS-backed import
# ============================================================
from utils.performance_setup import configure_performance  # noqa: E402

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)

# ============================================================
# SECTION 3: Heavy imports + flush-forcing print override (LESSONS_LEARNED_APPLIED.md #12)
# ============================================================
import builtins  # noqa: E402
import functools  # noqa: E402
import json  # noqa: E402
import warnings  # noqa: E402
from datetime import datetime, timezone  # noqa: E402

import polars as pl  # noqa: E402
import yaml  # noqa: E402

warnings.filterwarnings("ignore")
print = functools.partial(builtins.print, flush=True)

# ============================================================
# SECTION 4: BP1-BP7 upstream registry - the real bp_id/bp_name/config-path/artifacts-dir/
# reports-dir mapping this notebook checks live against. This table only names WHERE to look; it
# never asserts what it will find there - Section 5 below reads every one of these paths live.
# ============================================================
UPSTREAM_BPS = [
    {"bp_id": "bp1", "bp_name": "bp1_customer_intent_classification"},
    {"bp_id": "bp2", "bp_name": "bp2_customer_friction_classification"},
    {"bp_id": "bp3", "bp_name": "bp3_complaint_escalation_prediction"},
    {"bp_id": "bp4", "bp_name": "bp4_customer_journey_analytics"},
    {"bp_id": "bp5", "bp_name": "bp5_root_cause_driver_analytics"},
    {"bp_id": "bp6", "bp_name": "bp6_genai_resolution_assistant"},
    {"bp_id": "bp7", "bp_name": "bp7_customer_navigator_decision_engine"},
]

# Real Gold-layer Parquet tables a completed upstream BP has actually written to data/processed/,
# per that BP's own Gate 2 config block (bp2/bp3/bp4's own *_gold_path fields) or, for BP1, per
# Master Plan Section 6's shared "Common Taxonomy / Intent Layer" architecture description (BP1's
# own config carries no Gate 2 block - its Gate 2 output is this shared taxonomy layer, not a
# BP1-exclusive path - stated here as an ASSUMPTION, not read from a BP1-owned config field).
UPSTREAM_GOLD_TABLES = {
    "bp1": ["cfpb_common_taxonomy_gold.parquet", "banking77_common_taxonomy_gold.parquet"],
    "bp2": ["cfpb_friction_severity_gold.parquet"],
    "bp3": ["cfpb_intervention_escalation_gold.parquet"],
    "bp4": [
        "cfpb_journey_event_gold.parquet",
        "cfpb_issue_cluster_monthly_gold.parquet",
        "cfpb_issue_cluster_summary_gold.parquet",
    ],
    "bp5": [],
    "bp6": [],
    "bp7": [],
}

# ============================================================
# SECTION 5: Live check #1 - each upstream BP's own config status + policy.json existence (never
# assumed from this session's own prior conversation or from EVIDENCE_LEDGER.md's own narrative -
# read directly off this project's real config/artifact files, every run).
# ============================================================
upstream_bp_status = {}
for bp in UPSTREAM_BPS:
    bp_id, bp_name = bp["bp_id"], bp["bp_name"]
    config_path = CONFIGS_DIR / f"{bp_id}_{bp_name.split('_', 1)[1]}.yaml"
    policy_path = NOTEBOOKS_DIR / bp_name / "artifacts" / "policy.json"
    config_status = None
    if config_path.exists():
        with open(config_path, "r", encoding="utf-8") as f:
            config_yaml = yaml.safe_load(f)
        config_status = config_yaml.get("status") if isinstance(config_yaml, dict) else None
    gate6_reached = bool(config_status) and "gate6" in config_status.lower()
    upstream_bp_status[bp_id] = {
        "bp_name": bp_name,
        "config_yaml_exists": config_path.exists(),
        "config_status": config_status,
        "gate6_reached": gate6_reached,
        "policy_json_exists": policy_path.exists(),
    }
    print(f"[OK] {bp_id} ({bp_name}): status={config_status!r}, gate6_reached={gate6_reached}")

# ============================================================
# SECTION 6: Live check #2 - real Gold-layer Parquet tables. Read live via pl.scan_parquet (schema
# + row count only, no full materialization - WARP) for every table UPSTREAM_GOLD_TABLES names;
# never assumed present just because the owning BP's config says gate6_reached.
# ============================================================
for bp_id, filenames in UPSTREAM_GOLD_TABLES.items():
    tables = []
    for filename in filenames:
        table_path = DATA_PROCESSED_DIR / filename
        if not table_path.exists():
            tables.append({"filename": filename, "exists": False})
            continue
        lazy = pl.scan_parquet(table_path)
        n_rows = lazy.select(pl.len()).collect().item()
        columns = lazy.collect_schema().names()
        tables.append({
            "filename": filename,
            "exists": True,
            "n_rows": int(n_rows),
            "columns": list(columns),
        })
    upstream_bp_status[bp_id]["gold_tables"] = tables
    n_real = sum(1 for t in tables if t["exists"])
    print(f"[OK] {bp_id}: {n_real}/{len(filenames)} real Gold-layer table(s) found in data/processed/")

# ============================================================
# SECTION 7: Live check #3 - BP1-BP4's own per-BP Gate 7 executive-rollup output files (existence +
# size only - this notebook never opens another BP's DOCX/XLSX/PPTX; see the markdown cell's
# distinction section for why this is explicitly NOT BP8's own scope).
# ============================================================
ROLLUP_SUFFIXES = ["dashboard.html", "report.docx", "workbook.xlsx", "deck.pptx"]
for bp in UPSTREAM_BPS:
    bp_id, bp_name = bp["bp_id"], bp["bp_name"]
    rollup_dir = REPORTS_DIR / bp_name / "executive_rollup"
    files = []
    for suffix in ROLLUP_SUFFIXES:
        file_path = rollup_dir / f"{bp_id}_executive_rollup_{suffix}"
        files.append({
            "filename": file_path.name,
            "exists": file_path.exists(),
            "size_bytes": file_path.stat().st_size if file_path.exists() else 0,
        })
    upstream_bp_status[bp_id]["own_gate7_rollup_files"] = files
    n_present = sum(1 for fl in files if fl["exists"] and fl["size_bytes"] > 0)
    print(f"[OK] {bp_id}: {n_present}/{len(ROLLUP_SUFFIXES)} own Gate-7-rollup output file(s) present")

# ============================================================
# SECTION 8: Live check #4 - BP3's own real model artifact (models/bp3_.../*.joblib), the one
# upstream BP with a real trained/persisted model today, plus a directory listing for every other
# BP's models/ folder (documents real state, never assumed).
# ============================================================
for bp in UPSTREAM_BPS[:4]:
    bp_id, bp_name = bp["bp_id"], bp["bp_name"]
    model_dir = MODELS_DIR / bp_name
    real_model_files = []
    if model_dir.exists():
        real_model_files = [p.name for p in model_dir.iterdir() if p.is_file() and p.name != ".gitkeep"]
    upstream_bp_status[bp_id]["real_model_artifact_files"] = real_model_files
    print(f"[OK] {bp_id}: real model artifact file(s) in models/: {real_model_files or 'NONE'}")

# ============================================================
# SECTION 9: Live check #5 - the Power BI layer's own two artifact-format directories
# (powerbi/gold_tables/, powerbi/pbix/) - confirming, live, both are currently empty (BP8's own job
# has not started yet, matching every upstream BP5/6/7 config's own not_started status).
# ============================================================
powerbi_gold_tables_dir_exists = POWERBI_GOLD_TABLES_DIR.exists()
powerbi_pbix_dir_exists = POWERBI_PBIX_DIR.exists()
n_gold_table_files = (
    len([p for p in POWERBI_GOLD_TABLES_DIR.iterdir() if p.is_file()])
    if powerbi_gold_tables_dir_exists
    else 0
)
n_pbix_files = (
    len([p for p in POWERBI_PBIX_DIR.iterdir() if p.is_file()]) if powerbi_pbix_dir_exists else 0
)
print(
    f"[OK] powerbi/gold_tables/ exists={powerbi_gold_tables_dir_exists}, real files={n_gold_table_files} "
    f"(BP8's own Python-built Gold/semantic table output - not started yet)"
)
print(
    f"[OK] powerbi/pbix/ exists={powerbi_pbix_dir_exists}, real files={n_pbix_files} "
    f"(the .pbix you build in Power BI Desktop, per Section 19 - not started yet, not a Claude-authored "
    f"artifact)"
)

# ============================================================
# SECTION 10: Real KPI-category scope, grounded only in upstream BPs whose Gold-layer table
# live-verified to exist above (Section 6) - see the markdown cell's "Real KPI scope" section for
# the full field-by-field grounding. No KPI category is defined here for BP5/6/7 - all three
# live-verified not_started above.
# ============================================================
friction_severity_values = None
if upstream_bp_status["bp2"]["gold_tables"] and upstream_bp_status["bp2"]["gold_tables"][0]["exists"]:
    friction_table_path = DATA_PROCESSED_DIR / "cfpb_friction_severity_gold.parquet"
    friction_severity_values = (
        pl.scan_parquet(friction_table_path)
        .group_by("friction_severity_class")
        .agg(pl.len().alias("n"))
        .sort("n", descending=True)
        .collect()
        .to_dicts()
    )
    print(f"[OK] BP2 real friction_severity_class distribution (live): {friction_severity_values}")

intervention_required_values = None
if upstream_bp_status["bp3"]["gold_tables"] and upstream_bp_status["bp3"]["gold_tables"][0]["exists"]:
    escalation_table_path = DATA_PROCESSED_DIR / "cfpb_intervention_escalation_gold.parquet"
    intervention_required_values = (
        pl.scan_parquet(escalation_table_path)
        .group_by("intervention_required")
        .agg(pl.len().alias("n"))
        .sort("n", descending=True)
        .collect()
        .to_dicts()
    )
    print(f"[OK] BP3 real intervention_required distribution (live): {intervention_required_values}")

kpi_category_scope = {
    "friction_trends": {
        "source_bp": "bp2",
        "source_table": "data/processed/cfpb_friction_severity_gold.parquet",
        "source_field": "friction_severity_class",
        "real_live_value_distribution": friction_severity_values,
        "ready": upstream_bp_status["bp2"]["gate6_reached"],
    },
    "escalation_trends": {
        "source_bp": "bp3",
        "source_table": "data/processed/cfpb_intervention_escalation_gold.parquet",
        "source_field": "intervention_required",
        "real_live_value_distribution": intervention_required_values,
        "ready": upstream_bp_status["bp3"]["gate6_reached"],
    },
    "product_opportunity_flags": {
        "source_bp": "bp4",
        "source_table": "data/processed/cfpb_issue_cluster_summary_gold.parquet",
        "source_field": "review_priority_tier (BP4's own Gate 5 output, HIGH/MEDIUM/LOW/NONE)",
        "note": "BP8 aggregates BP4's own already-computed tier rollup; it never recomputes BP4's "
        "own flags or thresholds.",
        "ready": upstream_bp_status["bp4"]["gate6_reached"],
    },
    "customer_intent_and_volume": {
        "source_bp": "bp1",
        "source_table": "data/processed/cfpb_common_taxonomy_gold.parquet, "
        "data/processed/banking77_common_taxonomy_gold.parquet",
        "source_field": "category (77-class), common_taxonomy_bucket (9-bucket)",
        "ready": upstream_bp_status["bp1"]["gate6_reached"],
    },
    "root_cause_driver_kpis": {
        "source_bp": "bp5",
        "status": "DEFERRED - bp5 live-verified not_started, no Gold-layer output exists yet.",
        "ready": False,
    },
    "genai_resolution_kpis": {
        "source_bp": "bp6",
        "status": "DEFERRED - bp6 live-verified not_started, no Gold-layer output exists yet.",
        "ready": False,
    },
    "decision_engine_kpis": {
        "source_bp": "bp7",
        "status": "DEFERRED - bp7 live-verified not_started, no Gold-layer output exists yet.",
        "ready": False,
    },
}
n_kpi_categories_ready = sum(1 for v in kpi_category_scope.values() if v.get("ready"))
print(f"[OK] KPI categories ready today (live-verified): {n_kpi_categories_ready} of 7 defined")

# ============================================================
# SECTION 11: Assemble the Gate 1 policy - BP8's analog to BP1-BP3's target_definition/leakage_rules
# is aggregation_scope_definition/scope_boundaries (no predictive target exists - see markdown cell).
# ============================================================
policy = {
    "bp_id": "bp8",
    "bp_name": "bp8_executive_product_analytics",
    "gate": 1,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "aggregation_scope_definition": {
        "no_predictive_target": True,
        "naming_commitment": "BP8 is a cross-BP Gold-layer aggregation and Power BI reporting "
        "layer, never a ninth modeling problem - no target, no classifier, no GenAI call, per "
        "Master Plan Section 5.1/6/17.5.",
        "gate1_exit_criterion_reinterpreted": "Section 8's generic exit criterion ('No target "
        "leakage possible by construction') is trivially satisfied by construction for BP8: there "
        "is no predictive target for any feature to leak into. This is stated explicitly rather "
        "than left implicit or worked around with a fabricated target.",
        "power_bi_layer_scope": {
            "claude_authored_deliverable": "Gold/semantic tables written in Python to "
            "powerbi/gold_tables/ (Parquet/CSV or SQL-ready form), per Master Plan Section 19.",
            "not_claude_authored": "The interactive .pbix file itself - built by the user in Power "
            "BI Desktop, per Section 19's own words ('create the interactive .pbix in Power BI "
            "Desktop'). Never claimed as a BP8 notebook deliverable.",
            "powerbi_gold_tables_dir_exists": powerbi_gold_tables_dir_exists,
            "powerbi_gold_tables_real_files_today": n_gold_table_files,
            "powerbi_pbix_dir_exists": powerbi_pbix_dir_exists,
            "powerbi_pbix_real_files_today": n_pbix_files,
        },
        "bp1_4_rollup_vs_bp8_distinction": {
            "bp1_4_own_gate7_rollups": "Already real, already built per-BP (BP1/BP2/BP3/BP4 each "
            "have their own reports/bp{n}_.../executive_rollup/ HTML+DOCX+XLSX+PPTX set, "
            "live-verified in Section 7 below) - Master Plan Section 20's OPTIONAL, per-BP "
            "recruiter-facing Financial-Impact reporting layer. NOT BP8's job.",
            "bp8_power_bi_layer": "New, cross-BP, not yet started (live-verified below) - Master "
            "Plan Section 5/6/8/17.5/19's REQUIRED executive/product Power BI decision layer. This "
            "IS BP8's actual, sole job.",
        },
        "kpi_category_scope": kpi_category_scope,
    },
    "scope_boundaries": [
        "BP8 reads only already-aggregated Gold-layer columns upstream BPs' own gates already "
        "computed and governed (friction_severity_class, intervention_required, review_priority_"
        "tier, category/common_taxonomy_bucket) - never raw CFPB/BANKING77 rows directly.",
        "BP8 never recomputes an upstream BP's own classification, severity mapping, or tier "
        "logic - it aggregates and rolls up what that BP's own gates already produced.",
        "BP8 never opens or parses another BP's own Gate 7 rollup DOCX/XLSX/PPTX - those are that "
        "BP's own deliverable, not an input to BP8's Gold tables.",
        "No KPI category is defined for BP5/6/7 until that BP's own Gold-layer output live-"
        "verifies as present - no placeholder or illustrative KPI field is created now.",
        "BP8 authors Gold/semantic Python-built tables only (powerbi/gold_tables/); it never "
        "authors a .pbix file - that is a Power BI Desktop, human step (Section 19).",
    ],
    "explicitly_out_of_scope": [
        "Any per-customer/per-consumer rollup - BP8 inherits every upstream BP's own no-customer-"
        "identifier scope boundary (BP2/BP3/BP4 Gate 1 precedent); it aggregates at the same "
        "issue/event/cluster grain those BPs already established, never a person grain.",
        "Any financial-impact or dollar-value KPI not itself already a real, defensible figure "
        "computed by an upstream BP's own gates - Master Plan Section 20's dollar-shorthand rule "
        "applies if any such figure is ever added, never assumed here.",
        "Recomputing or overriding BP1-BP4's own already-confirmed Gate 3-6 metrics - BP8 reports "
        "them, it does not re-derive them.",
    ],
    "assumptions": [
        "BP1's own Gold-layer output (cfpb_common_taxonomy_gold.parquet, "
        "banking77_common_taxonomy_gold.parquet) is attributed here to BP1's Gate 2 (shared "
        "taxonomy/intent layer, Master Plan Section 6) - BP1's own config file carries no Gate 2 "
        "marker block of its own, so this attribution is stated as an ASSUMPTION grounded in the "
        "Master Plan's architecture description and the real file paths, not read from a "
        "BP1-owned config field.",
        "BP3's Gate 7 executive-rollup notebook was, as of this Gate 1's own live check, real-"
        "file-present on disk but not yet confirmed as the user's own real run in this project's "
        "Evidence Ledger - BP8 does not depend on BP3's Gate 7 rollup at all (see the "
        "bp1_4_rollup_vs_bp8_distinction field above), so this has no effect on BP8's own scope; "
        "stated here only for completeness.",
        "src/utils/bp1_config_sync.py is reused unmodified for BP8's own config file - already "
        "fully generic, parameterized by config_path, no BP1-specific logic (already reused "
        "unmodified by BP2, BP3 and BP4).",
        "This Gate 1 snapshot of upstream readiness (Section 5-9 above) is real as of this "
        "notebook's own run - BP8's future Gate 2 must re-check every one of these live checks "
        "again rather than trusting this Gate 1 artifact as still current.",
    ],
    "compliance_touchpoint": {
        "requirement": "Data-minimization & purpose-limitation statement (GLBA/GDPR-aligned, "
        "Master Plan Section 8's Gate 1 row)",
        "statement": "BP8 collects no new data of any kind - it reads only already-governed "
        "Gold-layer columns upstream BPs' own Gate 1-6 cycles already cleared for the stated "
        "purpose of executive/product analytics reporting (severity class, intervention flag, "
        "review-priority tier, intent bucket - never raw complaint narratives, never a "
        "demographic-adjacent field). No CFPB or BANKING77 row-level data is read directly by any "
        "BP8 gate. This satisfies the same data-minimization standard every upstream BP's own "
        "Gate 1 already established for its own scope.",
    },
    "live_checks": {
        "upstream_bp_status": upstream_bp_status,
        "powerbi_gold_tables_dir_exists": powerbi_gold_tables_dir_exists,
        "powerbi_gold_tables_real_files_today": n_gold_table_files,
        "powerbi_pbix_dir_exists": powerbi_pbix_dir_exists,
        "powerbi_pbix_real_files_today": n_pbix_files,
        "n_kpi_categories_ready_today": n_kpi_categories_ready,
        "n_kpi_categories_deferred": len(kpi_category_scope) - n_kpi_categories_ready,
    },
}

# ============================================================
# SECTION 12: Write outputs (idempotent overwrite-in-place)
# ============================================================
policy_json_path = ARTIFACTS_DIR / "policy.json"
with open(policy_json_path, "w", encoding="utf-8") as f:
    json.dump(policy, f, indent=2, default=str)
print(f"\n[SAVED] {policy_json_path.relative_to(PROJECT_ROOT)}")

bp8_config_path = CONFIGS_DIR / "bp8_executive_product_analytics.yaml"

# BP8 reuses BP1's marker-based config-sync helpers as-is (src/utils/bp1_config_sync.py) - already
# generic, already reused unmodified by BP2/BP3/BP4. Gate 1 here owns only the front-matter section
# below; no BP8 gate block exists yet (BP8 has not started before this notebook), so there is
# nothing downstream to preserve today - the marker-preserving behavior is exercised identically to
# every other BP's Gate 1 regardless.
from utils.bp1_config_sync import read_existing_gate_block_markers, write_front_matter  # noqa: E402

_existing_gate_markers = read_existing_gate_block_markers(bp8_config_path)
_status_suffix = ""
for _gate_num, _gate_label in ((2, "Gate 2"), (3, "Gate 3"), (4, "Gate 4"), (5, "Gate 5"), (6, "Gate 6")):
    if any(_gate_label in _m for _m in _existing_gate_markers):
        _status_suffix += f"_gate{_gate_num}_confirmed"

bp8_config_text = f"""# Per-BP config - filled in at Gate 1 (Business Understanding & Policy)
# Gate 1 owns bp_id through random_state below via write_front_matter() (src/utils/bp1_config_sync.py,
# reused as-is from BP1/BP2/BP3/BP4 - fully generic, parameterized by config_path); Gates 2-6 each
# own exactly one marker-delimited block appended after it via write_gate_block() - do not
# hand-edit either section, re-run the owning notebook instead.
bp_id: "bp8"
bp_name: "bp8_executive_product_analytics"
status: "gate1_confirmed{_status_suffix}"   # not_started|gate1|gate2|gate3|gate4|gate5|gate6_complete
aggregation_scope_definition:
  no_predictive_target: true
  naming_commitment: "Cross-BP Gold-layer aggregation and Power BI reporting layer, never a ninth
    modeling problem - no target, no classifier, no GenAI call (Master Plan Section 5.1/6/17.5)."
  power_bi_layer_scope:
    claude_authored_deliverable: "Gold/semantic tables written in Python to powerbi/gold_tables/
      (Parquet/CSV or SQL-ready form), per Master Plan Section 19."
    not_claude_authored: "The interactive .pbix file - built by the user in Power BI Desktop, per
      Section 19's own words. Never a BP8 notebook deliverable."
  bp1_4_rollup_vs_bp8_distinction:
    bp1_4_own_gate7_rollups: "Already real, already built per-BP (Section 20's OPTIONAL per-BP
      reporting layer) - NOT BP8's job."
    bp8_power_bi_layer: "New, cross-BP, not yet started - Section 5/6/8/17.5/19's REQUIRED
      executive/product Power BI decision layer. This IS BP8's actual, sole job."
scope_boundaries:
  - "BP8 reads only already-aggregated Gold-layer columns upstream BPs' own gates already computed
     and governed - never raw CFPB/BANKING77 rows directly."
  - "BP8 never recomputes an upstream BP's own classification, severity mapping, or tier logic - it
     aggregates and rolls up what that BP's own gates already produced."
  - "No KPI category is defined for BP5/6/7 until that BP's own Gold-layer output live-verifies as
     present - no placeholder or illustrative KPI field is created now."
  - "BP8 authors Gold/semantic Python-built tables only (powerbi/gold_tables/); it never authors a
     .pbix file - that is a Power BI Desktop, human step (Section 19)."
assumptions:
  - "BP1's Gold-layer output is attributed to BP1's Gate 2 (shared taxonomy/intent layer, Section
     6) as an ASSUMPTION - BP1's own config carries no Gate 2 marker block of its own."
  - "This Gate 1 snapshot of upstream readiness must be re-checked live by BP8's own future Gate 2 -
     never trusted as still current by then."
  - "src/utils/bp1_config_sync.py reused unmodified for BP8's own config file."
random_state: 42
"""
write_front_matter(bp8_config_path, bp8_config_text)
print(f"[SAVED] {bp8_config_path.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 13: Structural integrity checks - raise AssertionError, never silently pass
# ============================================================
checks = {
    "all_7_upstream_bps_checked": len(upstream_bp_status) == 7,
    "bp1_gold_tables_found": all(t["exists"] for t in upstream_bp_status["bp1"]["gold_tables"]),
    "bp2_gold_table_found": all(t["exists"] for t in upstream_bp_status["bp2"]["gold_tables"]),
    "bp3_gold_table_found": all(t["exists"] for t in upstream_bp_status["bp3"]["gold_tables"]),
    "bp4_gold_tables_found": all(t["exists"] for t in upstream_bp_status["bp4"]["gold_tables"]),
    "bp5_bp6_bp7_correctly_have_no_gold_tables_defined": all(
        len(upstream_bp_status[bp_id]["gold_tables"]) == 0 for bp_id in ("bp5", "bp6", "bp7")
    ),
    "no_kpi_category_defined_for_not_started_upstream_bp": all(
        kpi_category_scope[cat]["ready"] is False
        for cat in ("root_cause_driver_kpis", "genai_resolution_kpis", "decision_engine_kpis")
    ),
    "powerbi_gold_tables_dir_exists": powerbi_gold_tables_dir_exists,
    "powerbi_pbix_dir_exists": powerbi_pbix_dir_exists,
    "no_predictive_target_asserted": policy["aggregation_scope_definition"]["no_predictive_target"] is True,
    "policy_json_written": policy_json_path.exists(),
    "bp8_config_yaml_written": bp8_config_path.exists(),
}

print("\n=== INTEGRITY CHECKS ===")
for name, passed in checks.items():
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} {name}")
    assert passed, f"[CHECK FAILED] {name}"

print(
    f"\n[ALL CHECKS PASSED] BP8 Gate 1 complete - {n_kpi_categories_ready} of 7 KPI categories "
    "real-data-ready today (friction/escalation/product-opportunity/intent, grounded in BP1-BP4's "
    "own already-real Gold-layer columns), 3 correctly deferred (BP5/6/7, live-verified not_"
    "started). No predictive target exists for BP8, stated explicitly. powerbi/gold_tables/ and "
    "powerbi/pbix/ both confirmed empty - BP8's own Power BI Gold-layer build has not started, "
    "clearly separated from BP1-BP4's own already-real per-BP Gate 7 rollups. Proceed to BP8 "
    "Gate 2 (Data Verification & Gold-Table Aggregation) next, once at least one more upstream BP "
    "(BP5/6/7) is real, or with the four already-ready categories as an initial slice."
)
